# 02 – Building the SQL Database

**Objective:** move the raw CSV files into a relational SQLite database, then query it with SQL.

**Why a database:** the CSV files repeat the same information thousands of times. For example, each tournament's name and surface appear on every one of its matches. Splitting the data into linked tables removes that repetition and makes questions easier to ask.

## 1. Data model

| Table | One row per | Primary key | Main columns |
|---|---|---|---|
| `players` | player | `player_id` | name, hand, date of birth, country, height |
| `tournaments` | tournament edition | `tourney_id` | name, surface, level, draw size, start date |
| `matches` | match | `tourney_id` + `match_num` | round, score, both players' IDs, rankings, age and match statistics |

- **Primary key:** the column, or columns, that uniquely identify a row.
- **Foreign key:** a column pointing to another table's primary key. `matches.tourney_id` points to `tournaments`, while `matches.winner_id` and `matches.loser_id` point to `players`.
- **Where ranking and age are stored:** in `matches`, because they change over a career. Height and hand stay in `players`.

## 2. Load the raw files

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

RAW_DIR = Path("../data/raw")
DB_PATH = Path("../data/processed/tennis.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

files = sorted(RAW_DIR.glob("atp_matches_*.csv"))
raw = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
raw_players = pd.read_csv(RAW_DIR / "atp_players.csv", low_memory=False)

print(f"{len(raw):,} matches, {len(raw_players):,} players")

79,299 matches, 66,912 players


## 3. Split into three tables

SQLite has no dedicated date type, so dates are stored as `YYYY-MM-DD` text. In this format, sorting the text also sorts the dates correctly.

In [2]:
def to_iso_date(column):
    """Convert dates like 20240115 to '2024-01-15'. Invalid dates become missing."""
    as_text = column.astype("Int64").astype("string")
    return pd.to_datetime(as_text, format="%Y%m%d", errors="coerce").dt.strftime("%Y-%m-%d")


players = raw_players[["player_id", "name_first", "name_last", "hand", "dob", "ioc", "height"]].copy()
players["dob"] = to_iso_date(players["dob"])

tournaments = (
    raw[["tourney_id", "tourney_name", "surface", "tourney_level", "draw_size", "tourney_date"]]
    .drop_duplicates("tourney_id")
    .copy()
)
tournaments["tourney_date"] = to_iso_date(tournaments["tourney_date"])

# Everything that belongs to the tournament or to the player's profile is dropped from matches
moved_columns = [
    "tourney_name", "surface", "tourney_level", "draw_size", "tourney_date",
    "winner_name", "winner_hand", "winner_ht", "winner_ioc",
    "loser_name", "loser_hand", "loser_ht", "loser_ioc",
]
match_table = raw.drop(columns=moved_columns)

print(f"players: {players.shape}, tournaments: {tournaments.shape}, matches: {match_table.shape}")

players: (66912, 7), tournaments: (3757, 6), matches: (79299, 36)


## 4. Create the tables

`CREATE TABLE` defines each table's columns, data types and keys. With `PRAGMA foreign_keys = ON`, SQLite rejects any match that refers to a player or tournament that doesn't exist, which protects data quality.

In [4]:
SCHEMA = """
DROP TABLE IF EXISTS matches;
DROP TABLE IF EXISTS tournaments;
DROP TABLE IF EXISTS players;

CREATE TABLE players (
    player_id  INTEGER PRIMARY KEY,
    name_first TEXT,
    name_last  TEXT,
    hand       TEXT,
    dob        TEXT,
    ioc        TEXT,
    height     REAL
);

CREATE TABLE tournaments (
    tourney_id    TEXT PRIMARY KEY,
    tourney_name  TEXT NOT NULL,
    surface       TEXT,
    tourney_level TEXT,
    draw_size     INTEGER,
    tourney_date  TEXT NOT NULL
);

CREATE TABLE matches (
    tourney_id TEXT NOT NULL REFERENCES tournaments (tourney_id),
    match_num  INTEGER NOT NULL,
    round      TEXT,
    best_of    INTEGER,
    score      TEXT,
    minutes    REAL,

    winner_id INTEGER NOT NULL REFERENCES players (player_id),
    winner_seed REAL, winner_entry TEXT, winner_age REAL,
    winner_rank REAL, winner_rank_points REAL,

    loser_id INTEGER NOT NULL REFERENCES players (player_id),
    loser_seed REAL, loser_entry TEXT, loser_age REAL,
    loser_rank REAL, loser_rank_points REAL,

    w_ace REAL, w_df REAL, w_svpt REAL, w_1stIn REAL, w_1stWon REAL,
    w_2ndWon REAL, w_SvGms REAL, w_bpSaved REAL, w_bpFaced REAL,

    l_ace REAL, l_df REAL, l_svpt REAL, l_1stIn REAL, l_1stWon REAL,
    l_2ndWon REAL, l_SvGms REAL, l_bpSaved REAL, l_bpFaced REAL,

    PRIMARY KEY (tourney_id, match_num)
);
"""

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript(SCHEMA)

# Parent tables go first, because matches refers to them
players.to_sql("players", conn, if_exists="append", index=False)
tournaments.to_sql("tournaments", conn, if_exists="append", index=False)
match_table.to_sql("matches", conn, if_exists="append", index=False)
conn.commit()

## 5. Verify the load

A helper function runs a SQL query and returns the result as a DataFrame. The row counts below should match the shapes printed in section 3.

In [5]:
def q(sql):
    """Run a SQL query and return the result as a DataFrame."""
    return pd.read_sql_query(sql, conn)


q("""
SELECT 'players' AS table_name, COUNT(*) AS row_count FROM players
UNION ALL SELECT 'tournaments', COUNT(*) FROM tournaments
UNION ALL SELECT 'matches', COUNT(*) FROM matches
""")

,table_name,row_count
0,players,66912
1,tournaments,3757
2,matches,79299


## 6. SQL practice

Each section shows one example. Replace the placeholder query in each exercise with your own.

### 6.1 SELECT and LIMIT: choose columns and rows

In [7]:
q("""
SELECT name_first, name_last, ioc
FROM players
LIMIT 10
""")

,name_first,name_last,ioc
0,Gardnar,Mulloy,USA
1,Pancho,Segura,ECU
2,Frank,Sedgman,AUS
3,Giuseppe,Merlo,ITA
4,Richard,Gonzalez,USA
5,Grant,Golden,USA
6,Abe,Segal,RSA
7,Kurt,Nielsen,DEN
8,Istvan,Gulyas,HUN
9,Luis,Ayala,CHI


**Exercise 1.** Show the `tourney_name`, `surface` and `tourney_date` of 5 tournaments.

In [8]:
q("""
SELECT tourney_name, surface, tourney_date
FROM tournaments
LIMIT 5
""")

,tourney_name,surface,tourney_date
0,Auckland,Hard,2000-01-10
1,St. Poelten,Clay,2000-05-22
2,Munich,Clay,2000-05-01
3,Queen's Club,Grass,2000-06-12
4,Gstaad,Clay,2000-07-10


### 6.2 WHERE: filter rows

In [9]:
q("""
SELECT tourney_name, tourney_date
FROM tournaments
WHERE surface = 'Grass' AND tourney_date >= '2025-01-01'
""")

,tourney_name,tourney_date
0,s Hertogenbosch,2025-06-09
1,Stuttgart,2025-06-09
2,Halle,2025-06-16
3,Queen's Club,2025-06-16
4,Mallorca,2025-06-22
5,Eastbourne,2025-06-23
6,Wimbledon,2025-06-30
7,Davis Cup WG2 PO: PAK vs SEN,2026-02-06


**Exercise 2.** List all Grand Slam tournaments (`tourney_level = 'G'`) played on clay since 2020.

In [20]:
q("""
SELECT tourney_name, tourney_date, surface
FROM tournaments 
WHERE tourney_date >= '2020-01-01' AND tourney_level = 'G' AND surface = 'Clay'
""")

,tourney_name,tourney_date,surface
0,Roland Garros,2020-09-28,Clay
1,Roland Garros,2021-05-31,Clay
2,Roland Garros,2022-05-23,Clay
3,Roland Garros,2023-05-29,Clay
4,Roland Garros,2024-05-27,Clay
5,Roland Garros,2025-05-26,Clay
6,Roland Garros,2026-05-25,Clay


### 6.3 ORDER BY: sort results

`DESC` sorts from highest to lowest. Use `IS NOT NULL` to test for missing values, because `= NULL` never matches anything in SQL.

In [24]:
q("""
SELECT winner_id, winner_age, tourney_id
FROM matches
WHERE winner_age IS NOT NULL
ORDER BY winner_age DESC
LIMIT 5
""")

,winner_id,winner_age,tourney_id
0,103499,46.0,2026-M-DC-2026-WG2-PO-PAK-SEN-01
1,103499,44.6,2024-M-DC-2024-WG2-M-BAR-PAK-01
2,103499,43.6,2023-M-DC-2023-WG2-M-PAK-INA-01
3,103499,43.6,2023-M-DC-2023-WG2-M-PAK-INA-01
4,103529,43.4,2023-M-DC-2023-WG2-M-PAK-INA-01


**Exercise 3.** Find the 5 longest matches: show `tourney_id`, `round`, `score` and `minutes`.

In [27]:
q("""
SELECT tourney_id, round, score, minutes
FROM matches
WHERE score IS NOT NULL AND minutes IS NOT NULL
ORDER BY minutes DESC
LIMIT 5
""")

,tourney_id,round,score,minutes
0,2016-M001,QF,7-6(5) 6-3,1146.0
1,2017-0308,QF,6-4 3-6 6-2,987.0
2,2010-540,R128,6-4 3-6 6-7(7) 7-6(3) 70-68,665.0
3,2018-540,SF,7-6(6) 6-7(5) 6-7(9) 6-4 26-24,396.0
4,2004-520,R128,6-4 6-3 6-7(5) 3-6 16-14,393.0


### 6.4 GROUP BY: summarise groups

`GROUP BY` splits the rows into groups, and functions such as `COUNT` and `AVG` compute one value per group. It is the SQL equivalent of pandas `groupby`.

In [28]:
q("""
SELECT surface, COUNT(*) AS n_tournaments
FROM tournaments
GROUP BY surface
ORDER BY n_tournaments DESC
""")

,surface,n_tournaments
0,Hard,2059
1,Clay,1266
2,Grass,221
3,Carpet,196
4,None,15


**Exercise 4.** For each `round`, show the number of matches and the average `minutes`. Sort by average minutes, longest first.

In [29]:
q("""
SELECT round, COUNT(*) AS n_matches, AVG(minutes) AS avg_minutes
FROM matches
GROUP BY round
ORDER BY n_matches DESC
""")

,round,n_matches,avg_minutes
0,R32,24708,99.961986
1,R16,13432,101.917181
2,R64,11973,111.338013
3,R128,8672,136.302671
4,RR,8505,109.463021
5,QF,6755,103.680534
6,SF,3458,106.481404
7,F,1755,113.051402
8,ER,32,85.968750
9,BR,9,138.000000


### 6.5 JOIN: combine tables

`JOIN` links rows from two tables using a shared key. Here, each match is linked to its tournament.

In [ ]:
q("""
SELECT t.tourney_name, t.surface, m.round, m.score
FROM matches AS m
JOIN tournaments AS t ON m.tourney_id = t.tourney_id
WHERE t.tourney_name = 'Roland Garros' AND m.round = 'F'
ORDER BY t.tourney_date DESC
LIMIT 5
""")

**Exercise 5.** Find the 10 players with the most match wins since 2000. Join `matches` to `players` on `winner_id`, then group by player and count.

**Exercise 6 (harder).** Which 10 players have the most wins on clay? This needs **two** joins: one to `players` and one to `tournaments`.

In [ ]:
q("""
SELECT 'replace this query' AS exercise_5
""")

In [ ]:
q("""
SELECT 'replace this query' AS exercise_6
""")